<a href="https://colab.research.google.com/github/christinChris/gaiascope/blob/main/notebooks/BurnScope_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
import ee, geemap
from google.colab import auth

try:
    auth.authenticate_user()
    ee.Initialize(project="ee-christilekka")
    print("Connected to Earth Engine")
except Exception as e:
    print("Connection error:", e)

m = geemap.Map(center=[38.22, 23.93], zoom=12)
m.add_basemap("HYBRID")
m

Connected to Earth Engine


Map(center=[38.22, 23.93], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [101]:
aoi = ee.Geometry.Rectangle([23.80, 38.05, 24.05, 38.30])

before = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(aoi)
          .filterDate("2024-07-20", "2024-08-10")
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .median()
          .clip(aoi))

m.centerObject(aoi, 11)
m.addLayer(before, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Before · true color")
m

Map(center=[38.17499478165861, 23.92499999999964], controls=(WidgetControl(options=['position', 'transparent_b…

In [102]:
after = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(aoi)
         .filterDate("2024-08-15", "2024-09-05")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
         .median()
         .clip(aoi))

m.addLayer(after, {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}, "After · false color")
m

Map(center=[38.17499478165861, 23.92499999999964], controls=(WidgetControl(options=['position', 'transparent_b…

In [103]:
def s2_composite(start, end, area):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(area)
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .median()
            .clip(area))

aoi = ee.Geometry.Rectangle([23.75, 37.98, 24.10, 38.32])

before = s2_composite("2024-07-20", "2024-08-10", aoi)
after  = s2_composite("2024-08-15", "2024-09-05", aoi)

m = geemap.Map()
m.add_basemap("HYBRID")
m.centerObject(aoi, 11)
m.addLayer(before, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Before · true color")
m.addLayer(after,  {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}, "After · false color")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [104]:
def nbr(img):
    return img.normalizedDifference(["B8", "B12"]).rename("NBR")

dnbr  = nbr(before).subtract(nbr(after)).rename("dNBR")
burnt = dnbr.gt(0.27)

m.addLayer(dnbr, {"min": -0.1, "max": 0.8,
                  "palette": ["white", "yellow", "orange", "red", "black"]}, "dNBR")
m.addLayer(burnt.selfMask(), {"palette": ["#C2410C"]}, "Burnt (dNBR > 0.27)")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [105]:
lc   = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
land = lc.neq(80)

burnt = dnbr.gt(0.27).And(land)

m.addLayer(burnt.selfMask(), {"palette": ["#C2410C"]}, "Burnt · land only")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [106]:
burnt_m2 = (burnt.rename("burnt")
            .multiply(ee.Image.pixelArea())
            .reduceRegion(reducer=ee.Reducer.sum(),
                          geometry=aoi, scale=20, maxPixels=1e10)
            .get("burnt"))

km2 = ee.Number(burnt_m2).divide(1e6).getInfo()
print(f"Burnt area: {km2:.1f} km²")

Burnt area: 95.4 km²


In [107]:
import pandas as pd

classes = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
           50: "Built-up", 60: "Bare / sparse", 70: "Snow and ice",
           80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
           100: "Moss and lichen"}

groups = (ee.Image.pixelArea().updateMask(burnt)
          .addBands(lc)
          .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                        geometry=aoi, scale=20, maxPixels=1e10)
          .get("groups")
          .getInfo())

rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]

df = pd.DataFrame(rows)
df = df[df["Area (km²)"] >= 0.05]
df = df.sort_values("Area (km²)", ascending=False)
df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]

df.style.format(precision=1).hide(axis="index")

Land cover,Area (km²),Share (%)
Shrubland,48.1,50.5
Tree cover,35.5,37.2
Grassland,10.5,11.0
Cropland,0.6,0.7
Built-up,0.6,0.6
Total,95.3,100.0


In [108]:
SEVERITY   = {1: "Low", 2: "Moderate-low", 3: "Moderate-high", 4: "High"}
SEV_COLORS = ["#FFD54F", "#FF8F00", "#D84315", "#7F0000"]

def lc_table(groups):
    rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows)
    df = df[df["Area (km²)"] >= 0.05]
    df = df.sort_values("Area (km²)", ascending=False)
    df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
    df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]
    return df.style.format(precision=1).hide(axis="index")


def severity_df(groups):
    rows = [{"Land cover": classes[int(g["combo"]) // 10],
             "Severity":   SEVERITY[int(g["combo"]) % 10],
             "km²":        g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows).pivot_table(index="Land cover", columns="Severity",
                                        values="km²", aggfunc="sum", fill_value=0)
    df = df[[s for s in SEVERITY.values() if s in df.columns]]
    df = df[df.sum(axis=1) >= 0.05]
    return df.loc[df.sum(axis=1).sort_values().index]


def burnscope(aoi, before_dates, after_dates, threshold=0.27):
    # 1. Images
    before = s2_composite(before_dates[0], before_dates[1], aoi)
    after  = s2_composite(after_dates[0],  after_dates[1],  aoi)

    # 2. Science: dNBR, burnt mask (land only), severity classes
    dnbr  = nbr(before).subtract(nbr(after)).rename("dNBR")
    lc    = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
    burnt = dnbr.gt(threshold).And(lc.neq(80)).rename("burnt")
    sev   = (ee.Image(1).where(dnbr.gte(0.27), 2)
                        .where(dnbr.gte(0.44), 3)
                        .where(dnbr.gte(0.66), 4)
                        .updateMask(burnt).rename("severity"))

    # 3. Numbers: total, per land cover, per land cover × severity
    area  = ee.Image.pixelArea().updateMask(burnt)
    total = area.reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                              scale=20, maxPixels=1e10).get("area")
    groups = (area.addBands(lc)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                            geometry=aoi, scale=20, maxPixels=1e10)
              .get("groups"))
    combo = lc.toInt().multiply(10).add(sev).rename("combo")
    sev_groups = (area.addBands(combo)
                  .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="combo"),
                                geometry=aoi, scale=20, maxPixels=1e10)
                  .get("groups"))

    # 4. One trip to Google's servers
    info = ee.Dictionary({"total": total, "groups": groups, "sev_groups": sev_groups}).getInfo()

    return {"before": before, "after": after, "dnbr": dnbr, "burnt": burnt, "severity": sev,
            "km2": info["total"] / 1e6, "table": lc_table(info["groups"]),
            "sev_df": severity_df(info["sev_groups"])}

In [109]:
r = burnscope(aoi, ("2024-07-20", "2024-08-10"), ("2024-08-15", "2024-09-05"), threshold=0.44)

print(f"Burnt area: {r['km2']:.1f} km²")
r["table"]

Burnt area: 71.5 km²


Land cover,Area (km²),Share (%)
Shrubland,35.6,49.9
Tree cover,31.4,44.0
Grassland,4.0,5.7
Cropland,0.2,0.3
Built-up,0.2,0.3
Total,71.5,100.0


In [110]:
!pip install -q solara

In [111]:
from google.colab import output
output.enable_custom_widget_manager()

import solara, ipyleaflet
threshold = solara.reactive(0.27)

In [112]:
from google.colab import output
output.enable_custom_widget_manager()

In [113]:
import ipyleaflet
import matplotlib.pyplot as plt

before_start = solara.reactive("2024-07-20")
before_end   = solara.reactive("2024-08-10")
after_start  = solara.reactive("2024-08-15")
after_end    = solara.reactive("2024-09-05")
result = solara.reactive(None)
status = solara.reactive("")

basemap    = solara.reactive("Google satellite")
show_burnt = solara.reactive(True)

BASEMAPS = {
    "Google satellite": "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    "Google map":       "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    "Google terrain":   "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}",
}
TRUE_COLOR  = {"bands": ["B4", "B3", "B2"],  "min": 0, "max": 3000}
FALSE_COLOR = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}
SEV_LEGEND  = {"Low (0.10–0.27)":           "#FFD54F",
               "Moderate-low (0.27–0.44)":  "#FF8F00",
               "Moderate-high (0.44–0.66)": "#D84315",
               "High (> 0.66)":             "#7F0000"}
RESULT_LAYERS = ("Before · true color", "After · false color", "Burn severity (dNBR)")

m = geemap.Map(height="600px")
base = m.layers[0]
base.url = BASEMAPS[basemap.value]
base.name = "Basemap"
base.attribution = "Imagery © Google"
m.centerObject(aoi, 11)
m.add_legend(title="Burn severity (dNBR)", legend_dict=SEV_LEGEND, position="bottomright")

def set_basemap(name):
    base.url = BASEMAPS[name]

def remove_results():
    for c in list(m.controls):
        if isinstance(c, ipyleaflet.SplitMapControl):
            m.remove_control(c)
    for layer in list(m.layers):
        if layer.name in RESULT_LAYERS:
            m.remove_layer(layer)

def update_map(*args):
    r = result.value
    if r is None:
        return
    remove_results()
    left  = geemap.ee_tile_layer(r["before"], TRUE_COLOR,  "Before · true color")
    right = geemap.ee_tile_layer(r["after"],  FALSE_COLOR, "After · false color")
    m.add_layer(left)
    m.add_layer(right)
    m.add_control(ipyleaflet.SplitMapControl(left_layer=left, right_layer=right))
    m.addLayer(r["severity"], {"min": 1, "max": 4, "palette": SEV_COLORS},
               "Burn severity (dNBR)", show_burnt.value)

def clear_area(*args):
    remove_results()
    if m.draw_control is not None:
        m.draw_control.clear()
    result.value = None

def severity_chart(df):
    colors = [SEV_COLORS[list(SEVERITY.values()).index(c)] for c in df.columns]
    fig, ax = plt.subplots(figsize=(5, 3))
    df.plot(kind="barh", stacked=True, color=colors, ax=ax, width=0.7)
    ax.set_xlabel("Burnt area (km²)")
    ax.set_ylabel("")
    ax.legend(title="Severity", frameon=False, fontsize=8, title_fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig

def run():
    status.value = ""
    try:
        area = m.user_roi if m.user_roi is not None else aoi
        result.value = burnscope(area,
                                 (before_start.value, before_end.value),
                                 (after_start.value, after_end.value),
                                 threshold.value)
        update_map()
        m.centerObject(area)
    except Exception as e:
        status.value = f"⚠️ {e}"

@solara.component
def Page():
    with solara.Columns([1, 3]):
        with solara.Column():
            solara.Markdown("# 🔥 BurnScope")
            solara.Markdown("**When · before**")
            solara.InputText("From", value=before_start)
            solara.InputText("To", value=before_end)
            solara.Markdown("**When · after**")
            solara.InputText("From", value=after_start)
            solara.InputText("To", value=after_end)
            solara.SliderFloat("Burnt if dNBR above", value=threshold,
                               min=0.1, max=0.66, step=0.01, thumb_label="always")
            solara.Button("Map burnt area", on_click=run, color="primary")
            solara.Button("New area", on_click=clear_area, outlined=True)
            solara.Markdown(status.value)

            solara.Markdown("**Map layers**")
            solara.Select("Basemap", value=basemap, values=list(BASEMAPS), on_value=set_basemap)
            solara.Checkbox(label="Show burn severity (dNBR)", value=show_burnt, on_value=update_map)

            if result.value is not None:
                solara.Markdown(f"### 🔥 Burnt area: {result.value['km2']:.1f} km²")
                solara.display(result.value["table"])
                solara.Markdown("**Burnt area by land cover and severity**")
                solara.FigureMatplotlib(severity_chart(result.value["sev_df"]))
        with solara.Column():
            solara.display(m)

Page()

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [114]:
solara.Button("New area", on_click=clear_area, outlined=True)

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [116]:
m

Map(bottom=202255.0, center=[38.15, 23.925], controls=(WidgetControl(options=['position', 'transparent_bg'], p…